<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
# Máscaras BETO
[GitHub LINK](https://github.com/dccuchile/beto)

Importamos lo necesario

In [ ]:
import torch
from transformers import BertForMaskedLM, BertTokenizer

Definimos el modelo que usaremos

In [ ]:
model_name = "dccuchile/bert-base-spanish-wwm-uncased"

Cargamos el modelo con su tokenizador

In [ ]:
model = BertForMaskedLM.from_pretrained(model_name)
tokenizer = BertTokenizer.from_pretrained(model_name)

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/310 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31002, 768, padding_idx=1)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

Definimos un texto de ejemplo

In [ ]:
text = "Para solucionar los [MASK] del país, el presidente debe [MASK] de inmediato."

Tokenizamos

In [ ]:
inputs = tokenizer(text, return_tensors="pt")

input_ids = inputs["input_ids"][0]

Encontramos los MASKs

Cada token en BERT tiene un ID numérico único. El termino tokenizer.mask_token_id devuelve el ID asociado al token especial [MASK]. Este ID es el que el modelo reconoce como "posición a predecir".

In [ ]:
mask_token_id = tokenizer.mask_token_id

input_ids es un tensor con los IDs de todos los tokens de la oración.

Ejemplo:
* ["Para", "solucionar", "los", "[MASK]", ...] → [123, 456, 789, 103, ...]


(input_ids == mask_token_id)
Esto genera un tensor booleano:
* True en posiciones donde hay [MASK], False en el resto.

Ejemplo:
* [False, False, False, True, False, ..., True]

In [ ]:
# text = "FALSE FALSE FALSE [MASK] FALSE FALSE, FALSE FALSE FALSE [MASK] FALSE FALSE."

In [ ]:
masked_indices = (input_ids == mask_token_id).nonzero(as_tuple=True)[0]

In [ ]:
# Imprimimos el texto original (para referencia)
print("Texto:", text)

# Convertimos el tensor a lista para que sea más legible
# Ejemplo: tensor([3, 10]) a [3, 10]
print("Posiciones MASK:", masked_indices.tolist())

Texto: Para solucionar los [MASK] del país, el presidente debe [MASK] de inmediato.
Posiciones MASK: [4, 11]


Realizamos la predicción

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

Obtenemos el top-5

In [ ]:
top_k = 5

for i, idx in enumerate(masked_indices):
    # 0: primer ejemplo del batch
    # idx: posición del [MASK]
    mask_logits = logits[0, idx]

    top_tokens = torch.topk(mask_logits, top_k).indices
    predicted_tokens = tokenizer.convert_ids_to_tokens(top_tokens.tolist())

    print(f"\nMASK {i+1}:")
    for j, token in enumerate(predicted_tokens):
        print(f"{j+1}. {token}")


MASK 1:
1. problemas
2. asuntos
3. conflictos
4. temas
5. puntos

MASK 2:
1. actuar
2. intervenir
3. renunciar
4. partir
5. reunirse


Reconstruimos

In [ ]:
new_input_ids = input_ids.clone()

for idx in masked_indices:
    best_token = torch.argmax(logits[0, idx])
    new_input_ids[idx] = best_token

decoded = tokenizer.decode(new_input_ids, skip_special_tokens=True)

print("\nTexto reconstruido:")
print(decoded)


Texto reconstruido:
para solucionar los problemas del país, el presidente debe actuar de inmediato.
